# 04 · RT-DETR (query-based)

Rubric: *Query-based*. Same protocol and code path as `03_yolo.ipynb` (`ul_dataset.py`): 3-fold
GroupKFold by clip (`splits.json`), train frames every 20th frame, validation frames every 10th frame,
640 px, 12 epochs, OOF COCO metrics with ignored regions as `iscrowd=1` (`common.evaluate`).
Data-specific EDA (grid collisions, GT overlap, density vs queries) is in `03_yolo.ipynb`, section 4.2.

Before a `full` run, attach the output of the `03_yolo` notebook version that trained `yolo_base`
(**Add Input → Your Work → Notebooks**): its per-epoch history is used for the RT-DETR vs YOLO
convergence plot, and its `results.csv` for the comparison of all models.

`MODE`: `quick` (2 clips per fold part, 1 epoch, CPU is enough) · `full` (one experiment on all folds per Kaggle commit).

## 1. Run configuration

In [ ]:
MODE = "quick"              # "quick" | "full"
EXPERIMENT = "rtdetr_base"  # key of EXPERIMENTS (section 3), or a list: run one after another
FOLDS = [0, 1, 2]
REDO_FOLDS = []           # folds to retrain even if an attached output already has them (e.g. [2])

## 2. Setup
Install Ultralytics and get the shared modules (next to this notebook, an attached Kaggle Dataset, or the GitHub repo).

In [ ]:
import os, sys, shutil, subprocess, time, urllib.request
from pathlib import Path

os.environ["YOLO_VERBOSE"] = "False"   # no per-layer / per-batch printouts in the notebook
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--timeout", "60", "--retries", "5",
                    "ultralytics==8.4.203", "pycocotools"], check=True, timeout=1800)
    import ultralytics
os.environ["WANDB_MODE"] = "disabled"
try:   # faster-coco-eval: drop-in replacement of pycocotools, identical metrics, several times faster
    import faster_coco_eval
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--timeout", "60", "--retries", "5",
                    "faster-coco-eval"], timeout=900)
try:
    import faster_coco_eval
    faster_coco_eval.init_as_pycocotools()   # must run before `common` imports pycocotools
except Exception as e:
    print("faster-coco-eval unavailable, using pycocotools:", type(e).__name__)

REPO_RAW = "https://raw.githubusercontent.com/KaplyshOlha/hw-2/main/notebooks/"

def fetch(url, dst, timeout=60, tries=5):
    # download with a timeout and retries: a stalled GitHub request must not hang a commit for hours
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                Path(dst).write_bytes(r.read())
            return dst
        except Exception as e:
            if i == tries - 1:
                raise
            print(f"{url}: {type(e).__name__}, retry in 30 s"); time.sleep(30)
SRC_FILES = ["common.py", "ul_dataset.py", "splits.json"]
SRC = next((Path(p) for p in [".", "notebooks"] if all((Path(p) / f).exists() for f in SRC_FILES)), None)
if SRC is None:
    # the GitHub repo is the source of truth (always the latest commit); a copy in an attached
    # dataset is only a fallback for when GitHub is unreachable
    SRC = Path("/tmp/hw2_src"); SRC.mkdir(exist_ok=True)
    for f in SRC_FILES:
        try:
            fetch(REPO_RAW + f, SRC / f, tries=3); print(f"{f}: GitHub")
        except Exception:
            found = [p for pat in ["*", "*/*", "*/*/*"] for p in sorted(Path("/kaggle/input").glob(f"{pat}/{f}"))
                     if "competitions" not in p.parts] if Path("/kaggle/input").exists() else []
            if not found:
                raise
            shutil.copy(found[0], SRC / f); print(f"{f}: {found[0]} (GitHub unreachable)")
sys.path.insert(0, str(SRC))
print("ultralytics", ultralytics.__version__, "| sources:", SRC.resolve())

In [ ]:
import importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import common, ul_dataset as uld
importlib.reload(common); importlib.reload(uld)

assert MODE in ("quick", "full")
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
RUN_ROOT = WORK / ("ul_runs" if MODE != "quick" else "ul_runs_quick")
RESULTS = WORK / ("results.csv" if MODE != "quick" else "results_quick.csv")
DATA_ROOT = Path("/tmp/ul_data")
IMG_DIR = WORK / "images"
for p in (RUN_ROOT, IMG_DIR):
    p.mkdir(parents=True, exist_ok=True)
if MODE != "quick":
    assert torch.cuda.is_available(), "attach a GPU (Settings -> Accelerator)"
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
uld.restore_from_inputs(RUN_ROOT);   # finished RT-DETR folds and the yolo_base history from attached outputs

## 3. Experiments
Shared protocol as for YOLO (data, 640 px, 12 epochs, Ultralytics default augmentation with mosaic off
for the last 3 epochs, seed 42, `conf=0.001`, `max_det=300`, last-epoch weights). RT-DETR-specific:
`rtdetr-l` COCO weights (the smallest RT-DETR with pretrained weights in Ultralytics), AdamW,
lr0 = 1e-4, weight decay 1e-4, batch 8 (rtdetr-l at batch 16 does not fit a 16 GB T4 with AMP margin).
No NMS: the decoder outputs a set; `max_det` only caps the number of returned queries.

| experiment | change | motivation |
|---|---|---|
| `rtdetr_base` | rtdetr-l, 300 queries | reference (Ultralytics defaults) |
| `rtdetr_stable` | `warmup_bias_lr` 0.1 → 0 | `rtdetr_base` fold 2 collapsed: cls loss → 0 from epoch 2, precision/recall 0 for all 12 epochs while GIoU loss stayed ~2.2 (matched boxes barely overlap → IoU-aware class targets ≈ 0 → trivial all-zero scores). The default bias warmup starts at 10× lr0 in epoch 1 (meant for YOLO + SGD); removing it is the only change |
| `rtdetr_nq500` | `rtdetr_stable` + 500 decoder queries | queries vs scene density (EDA 4.2c): more queries than the densest frames, to check whether recall in crowded scenes is query-limited |
| `rtdetr_nq500_960` | `rtdetr_nq500` at 960 px input, batch 4 | input resolution, the strongest factor for YOLO (640 → 1024 px: +53 % mAP); RT-DETR at 640 px already matches YOLO at 1024 px (deviates from the 640 px protocol on purpose) |
| `rtdetr_nq500_960_s10` | `rtdetr_nq500_960` trained on every 10th frame instead of every 20th (2× frames and iterations, same 12 epochs) | training budget: validation mAP of RT-DETR was still rising at epoch 12 (section 6), so the 12-epoch / stride-20 protocol leaves it under-trained (final-pipeline run, deviates from the comparison protocol on purpose) |
| `rtdetr_nq500_nodn` | `rtdetr_nq500` without denoising queries (`nd` 100 → 0): no denoising loss | loss modification: the denoising task (DN-DETR) stabilises bipartite matching early in training; with 12 epochs it should matter, and its noised boxes are large relative to tiny objects |
| `rtdetr_nq500_giou` | `rtdetr_nq500` with box loss weights L1 5 → 2, GIoU 2 → 4 (matching costs unchanged) | loss modification: L1 on normalised coordinates gives tiny gradients for small boxes, GIoU is scale-invariant, so weight moves to the term that treats small objects like large ones |
| `rtdetr_bblr` | `rtdetr_stable` + backbone lr = 0.1 · lr0 (encoder/decoder keep lr0) | separate learning rates: keep COCO backbone features stable while the transformer adapts to 8 classes and tiny objects |

In [ ]:
BASE = uld.Config(experiment="rtdetr_base", model="rtdetr", weights="rtdetr-l.pt", device=DEVICE,
                  optimizer="AdamW", lr0=1e-4, weight_decay=1e-4, momentum=0.9, batch=8,
                  notes="rtdetr-l, 300 queries, AdamW 1e-4")
STABLE = uld.make_config(BASE, experiment="rtdetr_stable", warmup_bias_lr=0.0,
                         notes="rtdetr-l, no bias-lr warmup spike")
EXPERIMENTS = {
    "rtdetr_base": BASE,
    "rtdetr_stable": STABLE,
    "rtdetr_nq500": uld.make_config(STABLE, experiment="rtdetr_nq500", num_queries=500, notes="500 queries"),
    "rtdetr_nq500_960": uld.make_config(STABLE, experiment="rtdetr_nq500_960", num_queries=500, imgsz=960,
                                        batch=4, pred_batch=4, notes="500 queries, input 960 px (batch 4)"),
    "rtdetr_nq500_960_s10": uld.make_config(STABLE, experiment="rtdetr_nq500_960_s10", num_queries=500, imgsz=960,
                                            batch=4, pred_batch=4, train_stride=10,
                                            notes="500 queries, 960 px (batch 4), train stride 10"),
    "rtdetr_nq500_nodn": uld.make_config(STABLE, experiment="rtdetr_nq500_nodn", num_queries=500, num_denoising=0,
                                         notes="500 queries, no denoising queries"),
    "rtdetr_nq500_giou": uld.make_config(STABLE, experiment="rtdetr_nq500_giou", num_queries=500, rt_bbox_gain=2.0,
                                         rt_giou_gain=4.0, notes="500 queries, loss L1 x2 / GIoU x4"),
    "rtdetr_bblr": uld.make_config(STABLE, experiment="rtdetr_bblr", backbone_lr_mult=0.1,
                                   notes="backbone lr x0.1"),
}
RUN_LIST = list(EXPERIMENT) if isinstance(EXPERIMENT, (list, tuple)) else [EXPERIMENT]

def run_cfg(name):
    c = EXPERIMENTS[name]
    if MODE == "quick":
        c = uld.make_config(c, epochs=1, close_mosaic=0, batch=2, workers=2, fps_iters=3, fps_warmup=1, imgsz=320)
    return c

cfg = run_cfg(RUN_LIST[0])
print(cfg)

## 4. Data and folds
Same frames and folds as every other model; the converted frames are cached in `/tmp/ul_data`.

In [ ]:
common.seed_everything()
images, anns = common.load_gt(cache="/tmp/gt_cache.pkl")
splits = common.load_splits(SRC / "splits.json")
if MODE == "quick":
    for f in splits["folds"]:
        f["train_clips"], f["val_clips"] = f["train_clips"][:2], f["val_clips"][:2]
conv = uld.Converter(images, anns, DATA_ROOT)

## 5. Train and predict
Every fold in `FOLDS` (finished folds are skipped, an interrupted fold resumes from `last.pt`);
OOF metrics are logged to `results.csv` once all 3 folds are finished.

In [ ]:
t0 = time.time()
for name in RUN_LIST:   # one after another; `cfg` / `metrics` refer to the last one below
    cfg = run_cfg(name)
    metrics = uld.run_experiment(cfg, images, anns, splits, FOLDS, RUN_ROOT, RESULTS, conv, redo=REDO_FOLDS)
    print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)
print(f"{(time.time() - t0) / 60:.1f} min")

In [ ]:
MAIN = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500"]
PER_CLASS = [f"AP_{c}" for c in common.CLASSES]
if metrics is not None:
    display(metrics[MAIN].round(4))
    display(metrics.loc[["oof"], PER_CLASS].round(4))

## 6. Convergence: RT-DETR vs YOLO under the same budget
Validation mAP@[.5:.95] after every epoch on fold 0 (Ultralytics validator: same frames, same metric
for both models, but without ignored regions, so absolute values are below the OOF metric).
Same data, 12 epochs, 640 px. If the RT-DETR curve is still rising at epoch 12, it has not converged
within the budget.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
curves = {}
for e in ["yolo_base", *EXPERIMENTS]:
    done = uld.load_done(RUN_ROOT, e)
    if 0 in done and done[0]["history"]:
        curves[e] = pd.DataFrame(done[0]["history"])
for e, h in curves.items():
    style = "--" if e.startswith("yolo") else "-"
    ax[0].plot(h["epoch"], h["metrics/mAP50-95(B)"], style, marker="o", ms=3, label=e)
    ax[1].plot(h["epoch"], h["metrics/mAP50(B)"], style, marker="o", ms=3, label=e)
ax[0].set_title("val mAP50-95 (fold 0)"); ax[1].set_title("val mAP50 (fold 0)")
for a in ax:
    a.set_xlabel("epoch"); a.legend(fontsize=8)
fig.savefig(IMG_DIR / "04_rtdetr_vs_yolo_curves.png"); plt.show()
if "yolo_base" not in curves:
    print("yolo_base history not found: attach the 03_yolo notebook output (Add Input)")

**Observations (convergence):** TBD

## 7. Comparison of all models
Latest logged run of every experiment from this session, the attached `03_yolo` output and the
repo `results.csv` (teammate's torchvision models): OOF metrics on the same folds and frames.

In [ ]:
paths = [RESULTS, *uld._input_files("/kaggle/input", "results*.csv")]
try:
    fetch(REPO_RAW + "results.csv", "/tmp/repo_results.csv", tries=2); paths.append(Path("/tmp/repo_results.csv"))
except Exception as e:
    print("repo results.csv not available:", e)
frames = [pd.read_csv(p) for p in paths if Path(p).exists() and Path(p).stat().st_size
          and (MODE == "quick" or "quick" not in Path(p).name)]
allres = pd.concat(frames, ignore_index=True).astype({"fold": str}).drop_duplicates(["run_id", "fold"]) if frames else pd.DataFrame()
if len(allres):
    last = allres.sort_values("timestamp").groupby("experiment").run_id.last()
    r = allres[allres.run_id.isin(last) & (allres.fold == "oof")].set_index("experiment")
    std = allres[allres.run_id.isin(last) & (allres.fold == "std")].set_index("experiment")["mAP"]
    summ = r[["model", *MAIN, "fps", "train_min"]].assign(mAP_std=std).sort_values("mAP", ascending=False)
    display(summ.round(4))

**Observations (comparison):** TBD

## 8. Outputs
Merge this session's `results.csv` with the repo copy into `results_merged.csv` (commit it as `notebooks/results.csv`).

In [ ]:
if MODE == "full" and RESULTS.exists():
    merged = WORK / "results_merged.csv"
    try:
        fetch(REPO_RAW + "results.csv", merged, tries=2)
    except Exception as e:
        print("repo results.csv not available:", e); merged.unlink(missing_ok=True)
    common.merge_results(str(RESULTS), str(merged))
    print(f"{merged}: {len(pd.read_csv(merged))} rows")
for p in sorted(RUN_ROOT.glob("*/fold*/done.json")):
    print(p.parent)

## 9. Conclusions
TBD